<table style="width:100%; border:none;">
<tr style="border:none;">

<td style="width:20%; text-align:center; border:none;">
    <img src="https://raw.githubusercontent.com/Moises13r/PetroleumEng/refs/heads/main/Special%20Topics/Figures/logo_unam.png"
         width="120">
</td>

<td style="width:60%; text-align:center; border:none;">
    <h1>K-Nearest Neighbor</h1>
    <h2>Geospatial Deterministic Model</h2>
</td>

<td style="width:30%; text-align:center; border:none;">
    <img src="https://github.com/Moises13r/PetroleumEng/blob/main/Special%20Topics/Figures/logo_final_blanco.png?raw=true"
         width="150">
</td>

</tr>
</table>

*Authors:* 
* Nayely Teodoro Pérez
* Dr. Moisés Velasco Lozano

# Interpolación KNN multinivel: mapa espacial de permeabilidad

### Notebook · Versión didáctica

**Objetivo:** estimar la distribución espacial de permeabilidad \(k\) en una malla de **90 × 72 nodos (6,480 puntos)**, mediante el método **KNN multinivel con pesos de Sibson**, utilizando mediciones conocidas y optimizando la contribución del segundo nivel mediante validación cruzada.

### 🎯 Tu misión

A partir de las mediciones conocidas de permeabilidad:

1. Identificar los vecinos naturales mediante geometría de Voronoi.
2. Organizar los vecinos en dos niveles y calcular sus contribuciones.
3. Optimizar $\alpha_2$ mediante validación cruzada de cinco particiones.
4. Estimar la permeabilidad en la malla completa.
5. Generar el mapa espacial de permeabilidad y evaluar la precisión del método.

## 1. Instalación y bibliotecas

En este bloque se instalan e importan las bibliotecas necesarias para desarrollar la interpolación espacial de permeabilidad mediante el método **KNN multinivel con pesos de Sibson**.

### Bibliotecas utilizadas

1. **NumPy:** permite trabajar con arreglos numéricos y realizar los cálculos de permeabilidad.
2. **Pandas:** organiza los datos y facilita la creación de tablas de resultados.

3. **Matplotlib:** genera las gráficas y los mapas de distribución espacial de permeabilidad.

4. **SciPy:** permite localizar los vecinos más cercanos y analizar la distribución espacial de las mediciones.

5. **Shapely:** realiza operaciones geométricas con polígonos, áreas e intersecciones necesarias para calcular los pesos de Sibson.

6. **Scikit-learn:** proporciona herramientas para realizar la validación cruzada y evaluar los errores de las estimaciones.

7. **Openpyxl:** permite exportar los resultados a archivos de Excel.

8. **Ipywidgets:** proporciona controles interactivos para trabajar en Jupyter Notebook.

### Instalación y ejecución

- **Instalación:** se realiza mediante `%pip install`, que instala las bibliotecas que aún no están disponibles en el entorno.
- **Importación:** se realiza mediante `import` y `from`, para utilizar las funciones de cada biblioteca.
- **Verificación:** si todas las importaciones se ejecutan sin errores, las bibliotecas necesarias están disponibles.

**Nota:** las bibliotecas solo necesitan instalarse una vez por entorno de Python. Sin embargo, deben importarse cada vez que se reinicia el kernel y se ejecuta nuevamente el notebook.

In [ ]:
%pip install -q numpy pandas matplotlib scipy shapely scikit-learn openpyxl ipywidgets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial import cKDTree, ConvexHull
from shapely.geometry import Polygon, Point
from shapely.strtree import STRtree
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display
print("✓ Bibliotecas listas")

## 2. Selección y lectura del archivo

En este bloque se selecciona y carga el archivo `.npy` que contiene las mediciones conocidas de permeabilidad.

- **Formato:** archivo NumPy (`.npy`).
- **Contenido:** coordenadas espaciales \(x\), \(y\) y permeabilidad \(k\).
- **Selección:** mediante una ventana de búsqueda desde Jupyter Notebook.

El archivo contiene todas las observaciones disponibles para la interpolación, por lo que no se realiza una división inicial de los datos.

In [ ]:
from tkinter import Tk, filedialog
# Seleccionar archivo NPY
ventana = Tk()
ventana.withdraw()
ventana.attributes("-topmost", True)

archivo = filedialog.askopenfilename(
    title="Seleccionar archivo de permeabilidad",
    filetypes=[("Archivos NumPy", "*.npy")]
)
ventana.destroy()
if not archivo:
    raise ValueError("No se seleccionó ningún archivo.")
# Cargar las mediciones conocidas de permeabilidad
datos_npy = np.load(archivo)
# Organizar las mediciones en tres columnas
datos = pd.DataFrame(
    datos_npy,
    columns=["x (m)", "y (m)", "k (mD)"]
)
print(f"✓ Archivo cargado correctamente: {len(datos)} mediciones")
print("Archivo seleccionado:", archivo)

## 3. Comprobación de observaciones

Comprueba que las coordenadas y la permeabilidad sean numéricas, que no existan ubicaciones repetidas y que los valores de permeabilidad sean positivos. Muestra una vista previa y un mapa de los puntos conocidos.

In [ ]:
x = datos[_____].to_numpy(float)
y = datos[_____].to_numpy(float)
k = datos[_____].to_numpy(float)
P = np.column_stack([x, y])

print(f"✓ {len(P)} puntos conocidos cargados; k entre {k.min():.2f} y {k.max():.2f} mD")
from IPython.display import Markdown, display

# Vista previa de las observaciones originales y su identificación.
display(datos.head())
display(Markdown(
    "**Tabla 1.** Primeras cinco mediciones conocidas de permeabilidad, "
    "con sus coordenadas espaciales x, y (m) y permeabilidad k (mD)."))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

puntos = ax.scatter(_____, _____, c=_____, cmap="viridis", s=75, edgecolors="black", linewidths=.4)

fig.colorbar(puntos, ax=ax, label="Permeabilidad conocida k (mD)")
ax.set(xlabel="x (m)", ylabel="y (m)", title="Puntos conocidos")
ax.set_aspect("equal", adjustable="box")

# Pie de figura incorporado a la propia imagen, debajo de los ejes.
fig.subplots_adjust(bottom=0.21)
fig.text(0.5, 0.035, "Figura 1. Distribución espacial de las mediciones conocidas de permeabilidad.\n"
         "Cada punto indica una ubicación (x, y); el color representa k (mD).", ha="center", fontsize=9)
plt.show()

## 4. Construcción de la malla

La región de trabajo tiene **90 columnas × 72 filas = 6,480 nodos**. Se generan las coordenadas donde se estimará la permeabilidad, conservando las unidades espaciales en metros.

In [ ]:
nodos_x = np.arange(90, dtype=float)  # 0, 1, ..., 89 m
nodos_y = np.arange(72, dtype=float)  # 0, 1, ..., 71 m

X, Y = np.meshgrid(_____, _____)
Q = np.column_stack([X.ravel(), Y.ravel()])

assert X.shape == (72, 90) and len(Q) == 6480
print("✓ Malla creada:", X.shape, "=", len(Q), "nodos")

In [ ]:
Q.shape

## 5. Distancia euclidiana

La distancia permite identificar las observaciones próximas a cada punto objetivo. En este problema se utilizan dos dimensiones espaciales.

$$d_{iq}=\sqrt{\sum_{j=1}^{p}(x_{ij}-x_{qj})^2}$$

**Notación**

- $d_{iq}$: distancia entre la observación $i$ y el punto $q$.
- $x_{ij}$: coordenada $j$ de la observación $i$.
- $x_{qj}$: coordenada $j$ del punto objetivo $q$.
- $p$: número de dimensiones espaciales; aquí $p=2$.

**Fuente:** Cover y Hart (1967); distancia euclidiana aplicada a vecinos cercanos.

In [ ]:
def distancia_euclidiana(q, P_ent):
    return np.sqrt(np.sum((P_ent - q)**2, axis=1))

## 6. Geometría de Voronoi

Para calcular los pesos de Sibson, primero se construyen las regiones de influencia de cada punto conocido. El recorte geométrico permite obtener polígonos y calcular sus áreas. Este paso prepara la geometría; todavía no estima permeabilidad.

In [ ]:
def recortar_semiplano(poligono, normal, limite):
    """Recorta un polígono convexo por normal·x <= limite."""
    if len(poligono) == 0:
        return poligono
    salida = []
    anterior = poligono[-1]
    valor_ant = np.dot(anterior, normal) - limite
    for actual in poligono:
        valor_act = np.dot(actual, normal) - limite
        if (valor_ant <= 1e-10) != (valor_act <= 1e-10):
            t = valor_ant / (valor_ant - valor_act)
            salida.append(anterior + t * (actual - anterior))
        if valor_act <= 1e-10:
            salida.append(actual)
        anterior, valor_ant = actual, valor_act
    return np.array(salida, dtype=float).reshape(-1, 2)
def construir_geometria(P_ent):
    # El rectángulo amplio solo cierra las celdas no acotadas; no altera las áreas interiores.
    minimo = P_ent.min(axis=0)
    maximo = P_ent.max(axis=0)
    margen = max(float(np.ptp(P_ent, axis=0).max()), 1.0) * 10
    x0, y0 = minimo - margen
    x1, y1 = maximo + margen
    caja = np.array([[x0, y0], [x1, y0], [x1, y1], [x0, y1]], dtype=float)
    celdas = []
    for i, sitio in enumerate(P_ent):
        pol = caja.copy()
        for j, otro in enumerate(P_ent):
            if i == j:
                continue
            normal = otro - sitio
            limite = (np.dot(otro, otro) - np.dot(sitio, sitio)) / 2
            pol = recortar_semiplano(pol, normal, limite)
            if len(pol) < 3:
                break
        celdas.append(Polygon(pol) if len(pol) >= 3 else Polygon())
    return caja, celdas, STRtree(celdas), cKDTree(P_ent)

## 7. Pesos de Sibson

Al insertar el punto objetivo $q$, cada vecino natural cede parte de su región de Voronoi. El peso se calcula como la proporción de área cedida por ese vecino.

$$W_i(q)=\frac{A_i(q)}{\sum_{j\in N(q)}A_j(q)}$$

**Notación**

- $W_i(q)$: peso del vecino natural $i$ para estimar en $q$.
- $A_i(q)$: área cedida por el vecino $i$.
- $N(q)$: conjunto de vecinos naturales del punto $q$.
- $j$: índice de los vecinos incluidos en la suma.

**Fuente:** Sibson (1981), *A brief description of natural neighbour interpolation*, pp. 21–36.

In [ ]:
def pesos_sibson_2d(q, P_ent, geometria):
    caja, celdas, indice, arbol = geometria
    dist, cercano = arbol.query(q)
    if dist < 1e-9:  # El nodo coincide exactamente con una observación.
        return np.array([int(cercano)]), np.array([1.0]), "conocido"
    pol = caja.copy()
    for sitio in P_ent:
        normal = sitio - q
        limite = (np.dot(sitio, sitio) - np.dot(q, q)) / 2
        pol = recortar_semiplano(pol, normal, limite)
        if len(pol) < 3:
            return np.array([], dtype=int), np.array([]), "fuera"
    celda_q = Polygon(pol)
    candidatos = indice.query(celda_q)
    areas = np.array([celda_q.intersection(celdas[int(i)]).area for i in candidatos])
    buenos = areas > max(celda_q.area * 1e-11, 1e-10)
    vecinos = np.asarray(candidatos[buenos], dtype=int)
    areas = areas[buenos]
    if len(vecinos) == 0:
        return vecinos, areas, "fuera"
    return vecinos, areas / areas.sum(), "sibson"

## 8. Organización de los dos niveles

El **primer nivel** contiene los vecinos naturales de $q$. El **segundo nivel** contiene otros vecinos conocidos, seleccionados de manera distinta y próximos a los primeros. Se prepara una contribución de cada nivel para ajustar después su importancia.

$$\alpha_1=1$$

**Notación**

- $\alpha_1$: contribución de referencia del primer nivel.

**Fuente:** normalización de la adaptación metodológica KNN multinivel; no es una ecuación original de Sibson.

In [ ]:
def preparar_modelo(P_ent, z_ent):
    if len(P_ent) < 4:
        raise ValueError("Se necesitan al menos cuatro puntos de entrenamiento.")
    if np.linalg.matrix_rank(P_ent - P_ent.mean(axis=0)) < 2:
        raise ValueError("Los puntos deben ocupar un plano, no una sola línea.")
    return {"P": P_ent, "z": z_ent, "geo": construir_geometria(P_ent),
            "hull": Polygon(P_ent[ConvexHull(P_ent).vertices])}
def preparar_estimacion(q, modelo):
    P_ent, z_ent = modelo["P"], modelo["z"]
    arbol = modelo["geo"][3]
    distancia, exacto = arbol.query(q)
    if distancia < 1e-9:
        return float(z_ent[int(exacto)]), 0.0, "conocido"
    if not modelo["hull"].covers(Point(q)):
        _, idx = arbol.query(q)
        return float(z_ent[int(idx)]), float(z_ent[int(idx)]), "KNN exterior (k=1)"
    vecinos, pesos, estado = pesos_sibson_2d(q, P_ent, modelo["geo"])
    if len(vecinos) == 0:
        raise RuntimeError("No se pudieron calcular pesos de Sibson para un punto interior.")
    base1 = float(np.dot(pesos, z_ent[vecinos]))
    # Se selecciona un vecino exterior diferente para cada vecino natural.
    exteriores = [i for i in range(len(P_ent)) if i not in set(vecinos)]
    if len(exteriores) < len(vecinos):
        return base1, base1, "sibson sin nivel 2"
    disponibles = set(exteriores)
    asignados = []
    for i in vecinos:
        elegido = min(disponibles, key=lambda j: (np.linalg.norm(P_ent[j]-P_ent[i]), j))
        asignados.append(elegido)
        disponibles.remove(elegido)
    base2 = float(np.dot(pesos, z_ent[np.array(asignados)]))
    return base1, base2, "sibson multinivel"

## 9. Interpolación KNN multinivel

Se combinan las estimaciones de los dos niveles. Los pesos del primer nivel se reutilizan para ponderar el segundo, controlado por $\alpha_2$.

$$\hat z(q)=\frac{\sum_{i\in N_1(q)}W_i(q)z_{1,i}+\alpha_2\sum_{i\in N_1(q)}W_i(q)z_{2,i}}{1+\alpha_2}$$

**Notación**

- $\hat z(q)$: permeabilidad estimada en el punto $q$.
- $N_1(q)$: vecinos naturales del primer nivel.
- $W_i(q)$: peso de Sibson del vecino $i$.
- $z_{1,i}$: permeabilidad conocida del vecino del primer nivel.
- $z_{2,i}$: permeabilidad del vecino exterior asociado al vecino $i$.
- $\alpha_2$: contribución ajustable del segundo nivel.
- $q$: ubicación espacial donde se estima.

**Fuente:** adaptación algebraica de la interpolación ponderada y de los pesos de Sibson (1981); extensión multinivel del método estudiado.

**Límite:** fuera del contorno convexo no existen pesos de Sibson interiores; allí se utiliza exclusivamente el vecino conocido más cercano (KNN, $k=1$).

In [ ]:
def estimar_preparado(base1, base2, alpha_2):
    return (base1 + alpha_2 * base2) / (1 + alpha_2)

## 10. Validación cruzada y candidatos

Se evalúan **101 candidatos** de $\alpha_2$ mediante **5-Fold**: en cada vuelta se usan cuatro grupos para estimar y uno para validar. Se excluyen los puntos sin condiciones geométricas suficientes. Ningún dato se elimina del ajuste final.

$$\alpha_2\in\{0,0.01,0.02,\ldots,1.00\}$$

**Notación**

- $\alpha_2$: factor de contribución del segundo nivel.
- $0.01$: incremento entre candidatos.

**Fuente:** búsqueda por cuadrícula y validación cruzada, Hastie, Tibshirani y Friedman (2009); Stone (1974). Intervalo elegido para esta implementación.

In [ ]:
# # Evaluar valores positivos de alpha_2 para incluir ambos niveles


## 11. RMSE y NRMSE

El RMSE mide el error de predicción y penaliza especialmente las diferencias grandes. El NRMSE lo divide entre la dispersión de las mediciones reales, permitiendo evaluar los candidatos en una escala relativa.

$$RMSE=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(z_i-\hat z_i)^2}$$

**Notación**

- $RMSE$: raíz del error cuadrático medio.
- $n$: número de observaciones validadas.
- $z_i$: valor real de la observación $i$.
- $\hat z_i$: valor estimado para la observación $i$.
- $i$: índice de la observación.

**Fuente:** Chai y Draxler (2014), *Geoscientific Model Development*, 7, 1247–1250.

$$NRMSE=\frac{RMSE}{\sigma_z}$$

**Notación**

- $NRMSE$: error cuadrático medio normalizado.
- $\sigma_z$: desviación estándar de los valores reales de validación.
- $RMSE$: error definido anteriormente.

**Fuente:** normalización por desviación estándar aplicada a la calibración; RMSE según Chai y Draxler (2014).

In [ ]:
# *******************************************************

## 12. Selección de α₂

Se elige un **único** valor de $\alpha_2$ que minimiza el NRMSE calculado conjuntamente con los datos válidos de las cinco particiones. La gráfica permite ver si incorporar el segundo nivel reduce el error.

$$\alpha_2^*=\underset{\alpha_2\in\mathcal A}{\arg\min}\;NRMSE(\alpha_2)$$

**Notación**

- $\alpha_2^*$: valor óptimo seleccionado.
- $\alpha_2$: candidato evaluado.
- $\mathcal A$: conjunto de los 101 candidatos.
- $NRMSE(\alpha_2)$: error normalizado obtenido para ese candidato.
- $\arg\min$: operación que selecciona el argumento de menor error.

**Fuente:** selección de hiperparámetros mediante validación cruzada, Hastie, Tibshirani y Friedman (2009); aplicación particular a $\alpha_2$.

In [ ]:

alpha_2_optimo = float(0.01)


## 13. Evaluación de resultados

Además de RMSE y NRMSE, se calculan **MAE**, que expresa el error absoluto promedio, y **$R^2$**, que compara el ajuste con la variabilidad de los valores reales. La comparación real-estimado ayuda a detectar desviaciones. Estas métricas corresponden a validación cruzada, no a un conjunto de prueba independiente.

$$MAE=\frac1n\sum_{i=1}^n|z_i-\hat z_i|$$

**Notación**

- $MAE$: error absoluto medio.
- $n$: número de observaciones evaluadas.
- $z_i$: valor real.
- $\hat z_i$: valor estimado.

**Fuente:** Chai y Draxler (2014).

$$R^2=1-\frac{\sum_{i=1}^n(z_i-\hat z_i)^2}{\sum_{i=1}^n(z_i-\bar z)^2}$$

**Notación**

- $R^2$: coeficiente de determinación.
- $z_i$: valor real.
- $\hat z_i$: valor estimado.
- $\bar z$: media de los valores reales.
- $n$: número de observaciones.

**Fuente:** Draper y Smith (1998), *Applied Regression Analysis*, tercera edición.

In [ ]:
# ******************************************************

## 14. Estimación de la población completa

Con el parámetro óptimo se construye el modelo utilizando **todos los puntos conocidos** y se estiman los 6,480 nodos. Los valores observados se conservan en sus posiciones exactas. Se verifica que el resultado tenga **72 filas × 90 columnas**.

In [ ]:
%pip install tqdm

In [ ]:

from tqdm.auto import tqdm
import time

# Construir el modelo con todas las mediciones conocidas
inicio = time.perf_counter()
modelo_final = preparar_modelo(P, k)

# Estimar la permeabilidad en los 6,480 nodos
valores = np.empty(len(Q), dtype=float)
metodos = []

for j, q in enumerate(tqdm(Q, desc="Interpolando permeabilidad", unit="nodo")):
    base1, base2, metodo = preparar_estimacion(q, modelo_final)
    valores[j] = estimar_preparado(base1, base2, alpha_2_optimo)
    metodos.append(metodo)

# Organizar los resultados en la malla original
mapa = valores.reshape(72, 90)
metodos_mapa = np.asarray(metodos).reshape(72, 90)

# Verificar los resultados
assert mapa.shape == (72, 90) and np.isfinite(mapa).all()

print(f"✓ Nodos estimados: {len(Q)}")
print(f"✓ Dimensiones de la matriz: {mapa.shape}")
print(f"✓ Tiempo de cálculo: {(time.perf_counter() - inicio)/60:.2f} minutos")
print(pd.Series(metodos).value_counts())


## 15. Mapa de permeabilidad

Se dibuja un único mapa espacial de la permeabilidad estimada. Los círculos identifican las mediciones originales. En el interior del contorno se aplica Sibson multinivel; fuera, KNN con un vecino. La escala logarítmica facilita visualizar contrastes de permeabilidad.

In [ ]:
from matplotlib.colors import LogNorm
fig, ax = plt.subplots(figsize=(9, 6))


im = ax.pcolormesh(X, Y, mapa, cmap="viridis", shading="auto", norm=LogNorm(vmin=mapa.min(), vmax=mapa.max()))
ax.scatter(x, y, s=16, facecolors="none", edgecolors="white", linewidths=.7)
fig.colorbar(im, ax=ax, label="Permeabilidad estimada (mD)")
ax.set(xlabel="x (m)", ylabel="y (m)", title="KNN multinivel · 90 × 72")
ax.set_aspect("equal")


# Incorporar el pie antes de guardar, para incluirlo también en el PNG exportado.
fig.subplots_adjust(bottom=0.22)
fig.text(0.5, 0.035, "Figura 4. Distribución espacial estimada de permeabilidad en la malla de 90 × 72 nodos.\n"
         "Los círculos muestran las mediciones conocidas; el color indica k (mD) en escala logarítmica.",
         ha="center", fontsize=9)
fig.savefig("mapa_KNN_multinivel.png", dpi=200, bbox_inches="tight")
plt.show()
print("Nodos exteriores (KNN k=1):", sum(m == "KNN exterior (k=1)" for m in metodos))

## 16. Conclusiones

La interpolación **KNN multinivel con pesos de Sibson** permite estimar la distribución espacial de permeabilidad a partir de un conjunto reducido de mediciones conocidas.

Los resultados muestran que la precisión depende de la cantidad y distribución de los datos, así como de la selección de vecinos y la contribución del parámetro \(\alpha_2\).

**Este notebook es un punto de partida para seguir explorando el método.** Se invita a los estudiantes a experimentar con diferentes porcentajes de muestras, distribuciones de permeabilidad y criterios de selección de vecinos, comparando sus resultados mediante validación cruzada.